# 04 -- Collaborative Filtering (implicit ALS)

Matrix-factorization collaborative filtering via the **`implicit`** library
(`implicit.als.AlternatingLeastSquares`), evaluated with **precision@10**.

### Algorithm

Implicit-feedback ALS (Hu, Koren & Volinsky 2008):
1. Build a sparse user x song matrix from play counts; each cell is a
   confidence `c_ui = alpha * play_count`.
2. `AlternatingLeastSquares` alternates least-squares updates to extract
   latent user and song factors that best explain the play signal.
3. **4a -- user-based**: score every song for a user as `U_u . V_i`, recommend
   the top-10 the user hasn't played.
4. **4b -- item-based**: for a seed song, rank songs by cosine similarity of
   their latent profiles, recommend the top-10.

### Evaluation
- **Train/test split**: random holdout of 20% per user (no timestamps in
  the data, so "last 20%" is approximated by a random split).
- **Metric**: Precision@10 -- fraction of recommended tracks the user
  actually listened to in the test set. Target: **> 10%**.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-10, by likelihood score (descending) |
| `artist` | Artist name |
| `title` | Track title |

In [1]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd().parent))

from implicit.evaluation import train_test_split

from src.data import MySpotifyRecommender
from src.models.collaborative_filtering import (
    build_user_item_matrix,
    evaluate_user_cf,
    fit_als,
)


/home/samy/MySpotify/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rs = MySpotifyRecommender.from_files(
    data_dir=Path.cwd().parent / "data",
    download=True,
    # triplets_sample_rows=1_000_000
)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845943, 3)


---
## Research

### Train / Test Split

In [3]:
user_item, user_idx, song_idx, idx_song = build_user_item_matrix(rs)
user_item.shape

(2018374, 2018374)

In [4]:
train, test = train_test_split(user_item, train_percentage=0.8, random_state=42)

### 4. Collaborative Filtering

In [5]:
model = fit_als(train, factors=192, regularization=0.09, alpha=1.0, iterations=25)

/home/samy/MySpotify/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 28 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [03:00<00:00,  7.23s/it]


### 4a -- User-based recommendations (latent factors)

In [6]:
sample_user = ''
for i in user_idx.keys():
    sample_user = i
    break
print(f"Sample user: {sample_user}")

uid = user_idx[sample_user]

top_pred_indices, _ = model.recommend(
    uid, 
    train[uid],
    N=10,
    filter_already_liked_items=True
)

top_pred_item_ids = [idx_song[idx] for idx in top_pred_indices]

df = rs.tracks.filter(pl.col("song_id").is_in(top_pred_item_ids))
df = df.select(["artist", "title", "song_id"])
df = df.unique(subset=["song_id"])
df.select(["artist", "title"])

Sample user: b80344d063b5ccb3212f76538f3d9e43d87dca9e


artist,title
str,str
"""Maroon 5""","""Sunday Morning"""
"""Maroon 5""","""Won't Go Home Without You"""
"""Maroon 5""","""Wake Up Call"""
"""The Notting Hillbillies""","""Bewildered"""
"""Ramones""","""I Wanna Be Sedated (Remastered…"
"""Foo Fighters""","""The Pretender"""
"""Maroon 5""","""Makes Me Wonder"""
"""Foo Fighters""","""Everlong"""
"""Maroon 5 / Rihanna""","""If I Never See Your Face Again"""


### 4b -- Similar tracks (item-based CF)

In [7]:
top_song = top_pred_item_ids[1]

print(f"Top recommended song for user {sample_user}: {top_song}")

item_id = song_idx[top_song]

print(f"\n{rs.tracks.filter(pl.col('song_id') == top_song).select(['artist', 'title']).row(0, named=True)}")

similar_indices, similarity_scores = model.similar_items(itemid=item_id, N=11)
similar_indices = similar_indices[1:]
similarity_scores = similarity_scores[1:]

similar_item_ids = [idx_song[idx] for idx in similar_indices]

df = rs.tracks.filter(pl.col("song_id").is_in(similar_item_ids))
df = df.unique(subset=["song_id"])
df.select(["artist", "title"])

Top recommended song for user b80344d063b5ccb3212f76538f3d9e43d87dca9e: SOQLUTQ12A8AE48037

{'artist': 'Foo Fighters', 'title': 'The Pretender'}


artist,title
str,str
"""Foo Fighters""","""Everlong"""
"""Foo Fighters""","""Times Like These"""
"""Foo Fighters""","""All My Life"""
"""Banaroo""","""Space Cowboy"""
"""Foo Fighters""","""Learn To Fly"""
"""Björn Rosenström""","""Känner jag mig själv"""
"""Foo Fighters""","""Big Me"""
"""Maya Nasri""","""Wayli Aah"""
"""Foo Fighters""","""Breakout"""


#### Evaluation -- Precision@10

Pooled precision@10 over a random sample of test users. Measures what fraction of the top-10
recommended items appear in each user's held-out listening history.

- **ALS params**: factors=192, regularization=0.09, alpha=1.0, iterations=25
- **Train/test split**: 80/20 random
- **Target**: > 10%

In [8]:
pk_eval = evaluate_user_cf(
    model,
    train,
    test,
)

print(f"Precision@10: {pk_eval:.4f} ({pk_eval*100:.2f}%)")

Precision@10: 0.1181 (11.81%)
